# Pasado `gpu-hybrid`: verifikacija `planar_fit` (A2) na CUDA-i

Pre pokretanja izaberi **Runtime > Change runtime type > GPU (A100)**, zatim **Runtime > Disconnect and delete runtime**, pa **Runtime > Run all**.

A2 je izabran pre-registrovanim poređenjem (`results/lstsq_comparison_cuda_2026-10-27/README.md`) i uključen u batched put iza prekidača `batched_splits.USE_CLOSED_FORM_FIT` (podrazumevano isključen). Ova sveska proverava ga na A100 pre odluke da prekidač postane podrazumevano uključen.

1. GPU (A100) i kloniranje `gpu-hybrid` na fiksnom commitu `c2dcc4e`.
2. `test_planar_fit.py --device cuda`: isti bitovi kao validirani kandidat, ugovor C1-C7, blizina `gelsd`-u.
3. `test_batched_splits.py --device cuda`, sa `gelsd` i sa `--closed-form`: svaka granica naspram sekvencijalne petlje, autograd, i provera da je izabrani solver stvarno pozvan.
4. **Brzina u istom runu**: sekvencijalna petlja, batched sa `gelsd`, batched sa `planar_fit`; 4 mreže, `n_splits = 1, 8, 32, 64`, 3 slike, `eps_idx` 0 i 8. Broj poziva `fit_planes` se proverava u svakom režimu.
5. **Udeo `planar_fit.fit_planes`** u batched prolazu na `big`.
6. Kriterijumi iz briefa: `big` najmanje 10x brži od sekvencijalne petlje na `n_splits = 32/64`, udeo solvera ispod 20%.
7. Zip za preuzimanje.

Očekivano trajanje: oko 15 do 30 minuta. Kad završi, pošalji `planar_fit_cuda.zip`.

In [ ]:
import subprocess
import torch

subprocess.run(["nvidia-smi"], check=True)
assert torch.cuda.is_available(), "Uključi GPU: Runtime > Change runtime type > GPU"
GPU = torch.cuda.get_device_name(0)
VRAM_GIB = torch.cuda.get_device_properties(0).total_memory / 2**30
print("Torch:", torch.__version__, "| CUDA:", torch.version.cuda, "| GPU:", GPU,
      "| VRAM GiB:", round(VRAM_GIB, 1))
REQUIRE_A100 = True
assert "A100" in GPU or not REQUIRE_A100, f"Merenje je dogovoreno na A100, a ovo je {GPU}."

In [ ]:
import os
import shutil

REPO = "/content/Pasado"
URL = "https://github.com/dimitrijepesic/Pasado.git"
COMMIT = "c2dcc4e"  # planar_fit behind PASADO_CLOSED_FORM_FIT, with its tests

os.chdir("/content")
shutil.rmtree(REPO, ignore_errors=True)
subprocess.run(["git", "-c", "http.version=HTTP/1.1", "clone", "--branch", "gpu-hybrid",
                "--single-branch", URL, REPO], check=True)
subprocess.run(["git", "-C", REPO, "checkout", "-q", COMMIT], check=True)
head = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()
assert head.startswith(COMMIT), f"HEAD je {head}, a ocekivan je {COMMIT}"
print("HEAD:", head)
os.chdir(REPO)

OUT = "/content/planar_fit_cuda"
os.makedirs(OUT, exist_ok=True)

In [ ]:
%pip install -q scikit-learn mpmath

In [ ]:
def run(script_and_args, log):
    """A test script with its output on screen and in a log file; returns the exit code."""
    cmd = f'''
set -o pipefail
PYTHONUNBUFFERED=1 python {script_and_args} 2>&1 | tee {log}
'''
    return subprocess.run(["bash", "-lc", cmd], cwd=REPO).returncode

## 1. `test_planar_fit.py` na CUDA-i

In [ ]:
rc = run("experiments/test_planar_fit.py --device cuda", f"{OUT}/test_planar_fit_cuda.log")
assert rc == 0, "test_planar_fit nije prošao; ne nastavljaj, pošalji test_planar_fit_cuda.log"

## 2. `test_batched_splits.py` na CUDA-i, sa `gelsd` i sa `planar_fit`

Pun test (sve 4 mreže). Granica za svaki split mora da se slaže sa sekvencijalnom petljom do `1e-10`, a poslednja provera potvrđuje koji solver je stvarno korišćen.

In [ ]:
rc = run("experiments/test_batched_splits.py --device cuda", f"{OUT}/test_batched_splits_gelsd_cuda.log")
assert rc == 0, "test_batched_splits (gelsd) nije prošao; pošalji log"
rc = run("experiments/test_batched_splits.py --device cuda --closed-form",
         f"{OUT}/test_batched_splits_closed_form_cuda.log")
assert rc == 0, "test_batched_splits --closed-form nije prošao; pošalji log"

## 3. Brzina u istom runu: sekvencijalno, batched + `gelsd`, batched + `planar_fit`

Sva tri režima na istoj mašini, istim slikama i epsilonima, seed 7. Medijana od 3 ponavljanja posle jednog warm-up-a, `torch.cuda.synchronize()` oko svakog. U svakom režimu se broje pozivi `planar_fit.fit_planes`: mora biti 0 u `gelsd` režimima i više od 0 u `planar_fit` režimu. Granice se porede sa sekvencijalnom petljom.

In [ ]:
import math
import statistics
import sys
import time

import pandas as pd
from IPython.display import display

for p in (f"{REPO}/forward_mode_tensorized_src", f"{REPO}/Section_5_4", f"{REPO}/experiments"):
    if p not in sys.path:
        sys.path.insert(0, p)

import batched_splits as bs
import planar_fit
import precise_transformer as pt
from compare_lstsq_solvers import EPS_SCHEDULE, set_production_flags
from measure_split_sweep import build_forward_precise, eval_splits_once, load_network_and_images

set_production_flags()
torch.set_default_dtype(torch.float64)
device = torch.device("cuda")
SPLITS = [1, 8, 32, 64]
EPS_INDICES = [0, 8]
NUM_IMAGES = 3
REPEATS = 3
# about 0.5 GB per split on big in float64; keep to 60% of VRAM
BIG_CHUNK = min(64, 2 ** int(math.log2(max(1, 0.6 * VRAM_GIB / 0.5))))
print("chunk for big:", BIG_CHUNK)

# count the calls of the closed form, to see which solver each mode really uses
fit_planes = planar_fit.fit_planes
calls = [0]


def counted_fit_planes(grid, z, rcond=None):
    calls[0] += 1
    return fit_planes(grid, z, rcond)


planar_fit.fit_planes = counted_fit_planes


def timed(fn):
    fn()  # warm-up
    times = []
    for _ in range(REPEATS):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        value = fn()
        torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000.0)
    return statistics.median(times), value


rows = []
for network in ["3layer", "4layer", "5layer", "big"]:
    net, images = load_network_and_images(network, device, torch.float64, NUM_IMAGES)
    forward_fn = build_forward_precise(pt, net, network)
    layers = bs.extract_affine_layers(net, network)
    chunk = BIG_CHUNK if network == "big" else None
    for image_id, img in images:
        for eps_idx in EPS_INDICES:
            eps = EPS_SCHEDULE[eps_idx]
            for n_splits in SPLITS:
                row = dict(network=network, image_id=image_id, eps_idx=eps_idx, n_splits=n_splits)
                for mode in ("sequential", "batched_gelsd", "batched_planar_fit"):
                    bs.USE_CLOSED_FORM_FIT = mode == "batched_planar_fit"
                    calls[0] = 0
                    if mode == "sequential":
                        fn = lambda: eval_splits_once(forward_fn, img, eps, n_splits, 7, device,
                                                      torch.float64)
                    else:
                        fn = lambda: bs.lipschitz_bound(layers, img, eps, n_splits, seed=7,
                                                        chunk=chunk)
                    ms, bound = timed(fn)
                    used = calls[0] > 0
                    assert used == (mode == "batched_planar_fit"), (
                        f"{mode}: planar_fit pozvan {calls[0]} puta")
                    row[f"ms_{mode}"] = ms
                    row[f"bound_{mode}"] = bound
                rows.append(row)
            print(f"{network} image {image_id} eps_idx {eps_idx} done", flush=True)
bs.USE_CLOSED_FORM_FIT = False

df = pd.DataFrame(rows)
for mode in ("batched_gelsd", "batched_planar_fit"):
    df[f"rel_diff_{mode}"] = (df[f"bound_{mode}"] - df["bound_sequential"]).abs() / df["bound_sequential"].abs()
df.to_csv(f"{OUT}/speed_same_run.csv", index=False)
worst = df[["rel_diff_batched_gelsd", "rel_diff_batched_planar_fit"]].max()
print("largest relative difference to the sequential bound:")
print(worst.to_string())
assert (worst <= 1e-9).all(), "batched granica se ne slaže sa sekvencijalnom; pošalji speed_same_run.csv"

summary = (df.groupby(["network", "eps_idx", "n_splits"], as_index=False)
             .agg(ms_sequential=("ms_sequential", "mean"),
                  ms_batched_gelsd=("ms_batched_gelsd", "mean"),
                  ms_batched_planar_fit=("ms_batched_planar_fit", "mean")))
summary["speedup_planar_fit_vs_sequential"] = summary.ms_sequential / summary.ms_batched_planar_fit
summary["speedup_planar_fit_vs_batched_gelsd"] = summary.ms_batched_gelsd / summary.ms_batched_planar_fit
summary["speedup_batched_gelsd_vs_sequential"] = summary.ms_sequential / summary.ms_batched_gelsd
summary.to_csv(f"{OUT}/speed_same_run_summary.csv", index=False)
display(summary.round(2))

## 4. Udeo `planar_fit.fit_planes` u batched prolazu na `big`

Poseban prolaz (1 slika, `eps_idx = 0`) sa `torch.cuda.synchronize()` oko svakog poziva `fit_planes`, da bi se izmerilo samo njegovo vreme; ti sync-ovi malo usporavaju ovaj prolaz, pa se ukupno vreme uzima iz istog prolaza.

In [ ]:
net, images = load_network_and_images("big", device, torch.float64, 1)
layers = bs.extract_affine_layers(net, "big")
img = images[0][1]
eps = EPS_SCHEDULE[0]
acc = [0.0]


def timed_fit_planes(grid, z, rcond=None):
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    out = fit_planes(grid, z, rcond)
    torch.cuda.synchronize()
    acc[0] += (time.perf_counter() - t0) * 1000.0
    return out


planar_fit.fit_planes = timed_fit_planes
bs.USE_CLOSED_FORM_FIT = True
share = []
for n_splits in SPLITS:
    bs.lipschitz_bound(layers, img, eps, n_splits, seed=7, chunk=BIG_CHUNK)  # warm-up
    totals, fits = [], []
    for _ in range(REPEATS):
        acc[0] = 0.0
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        bs.lipschitz_bound(layers, img, eps, n_splits, seed=7, chunk=BIG_CHUNK)
        torch.cuda.synchronize()
        totals.append((time.perf_counter() - t0) * 1000.0)
        fits.append(acc[0])
    t, f = statistics.median(totals), statistics.median(fits)
    share.append(dict(n_splits=n_splits, total_ms=t, fit_planes_ms=f, share_pct=100.0 * f / t))
bs.USE_CLOSED_FORM_FIT = False
planar_fit.fit_planes = fit_planes
share = pd.DataFrame(share)
share.to_csv(f"{OUT}/planar_fit_share_big.csv", index=False)
display(share.round(2))

## 5. Kriterijumi iz briefa (informativno)

`big`: batched sa `planar_fit` najmanje 10x brži od sekvencijalne petlje na `n_splits = 32` i `64` (oba epsilona), udeo `fit_planes` ispod 20%. Male mreže ne smeju da budu sporije nego sa batched `gelsd`.

In [ ]:
big = summary[(summary.network == "big") & (summary.n_splits.isin([32, 64]))]
print(big[["eps_idx", "n_splits", "speedup_planar_fit_vs_sequential"]].round(2).to_string(index=False))
ok_speed = bool((big.speedup_planar_fit_vs_sequential >= 10).all())
ok_share = bool((share[share.n_splits.isin([32, 64])].share_pct < 20).all())
small = summary[summary.network != "big"]
ok_small = bool((small.speedup_planar_fit_vs_batched_gelsd >= 0.95).all())
print(f"big >= 10x vs sequential at 32/64 splits: {'PASS' if ok_speed else 'FAIL'}")
print(f"fit_planes share < 20% at 32/64 splits:    {'PASS' if ok_share else 'FAIL'}")
print(f"small networks not slower than batched gelsd (within 5%): {'PASS' if ok_small else 'FAIL'}")

## 6. Preuzimanje rezultata (`.zip`)

In [ ]:
import json
from zipfile import ZIP_DEFLATED, ZipFile
from google.colab import files

with open(f"{OUT}/colab_environment.json", "w") as f:
    json.dump(dict(gpu=GPU, vram_gib=VRAM_GIB, torch=torch.__version__, cuda=torch.version.cuda,
                   commit=head, big_chunk=BIG_CHUNK, repeats=REPEATS), f, indent=2)

archive = "/content/planar_fit_cuda.zip"
with ZipFile(archive, "w", ZIP_DEFLATED) as z:
    for name in sorted(os.listdir(OUT)):
        z.write(os.path.join(OUT, name), os.path.join("planar_fit_cuda", name))
print(archive, round(os.path.getsize(archive) / 2**20, 2), "MiB")
files.download(archive)